# Tarea 2: Fundamentos de Python
## Ciencia de Datos Ambientales - UTEC

**Nombre:** ADRIANO NICOLAS RODRIGUEZ BENDRELL  
**Puntaje total:** 20 puntos

**Instrucciones:**
- Completa todos los problemas en este notebook
- Escribe tu codigo en las celdas proporcionadas
- Ejecuta todas las celdas antes de entregar
- Sube el archivo `.ipynb` completado al modulo correspondiente en Canvas

**Integridad academica:** Tarea individual. Puedes consultar materiales del curso y documentacion de Python, pero todo el codigo debe ser tuyo.

---

## Problema 1: Procesador de Nombres de Archivos Landsat (10 puntos)

Trabajas con imagenes satelitales Landsat del Peru. Los nombres siguen el formato:

```
LC08_L2SP_008067_20240615_02_T1_SR_B4.TIF
```
Componentes: `{sensor}_{nivel}_{path_row}_{fecha}_{coleccion}_{tier}_SR_{banda}.TIF`

> Los paths 003-009, filas 062-071 cubren el territorio peruano (Madre de Dios, Loreto, Lima, Cusco).

### Tus tareas:

**Parte A (4 pts):** Funcion `procesar_nombre_landsat(nombre_archivo)` que devuelva un diccionario con:
- `sensor` (ej. "LC08"), `path` (ej. "008"), `row` (ej. "067")
- `fecha` formateada como "AAAA-MM-DD"
- `banda` (ej. "B4")

**Parte B (3 pts):** Funcion `clasificar_banda(banda)` que devuelva el nombre segun la tabla:

| Banda | Nombre |
|-------|--------|
| B1 | Aerosol costero | B2 | Azul | B3 | Verde | B4 | Rojo |
| B5 | Infrarrojo cercano (NIR) | B6 | SWIR1 | B7 | SWIR2 |

Si no esta en la tabla, devuelve "Desconocida".

**Parte C (3 pts):** Procesa la lista de archivos: parsea, imprime resumen (fecha/path/row/banda) y cuenta cuantas fechas unicas hay.

In [10]:
# Archivos Landsat sobre el Peru (paths 008-009: Madre de Dios, Ucayali, Loreto)
archivos = [
    "LC08_L2SP_008067_20240615_02_T1_SR_B4.TIF",
    "LC08_L2SP_008067_20240615_02_T1_SR_B5.TIF",
    "LC08_L2SP_008067_20240701_02_T1_SR_B3.TIF",
    "LC08_L2SP_009067_20240615_02_T1_SR_B4.TIF",
    "LC09_L2SP_008067_20240708_02_T1_SR_B6.TIF",
    "LC08_L2SP_008067_20240701_02_T1_SR_B4.TIF",
]

# Parte A: funcion procesar_nombre_landsat

def procesar_nombre_landsat(nombre_archivo):
  base=nombre_archivo.replace(".TIF","").replace(".tif", "")
  partes=base.split("_")
  sensor=partes[0]
  path_row=partes[2]
  path=path_row[:3]
  row=path_row[3:]
  fecha_raw=partes[3]
  fecha_formt=f"{fecha_raw[:4]}-{fecha_raw[4:6]}-{fecha_raw[6:]}"
  banda=partes[7]
  return {
        "sensor": sensor,
        "path": path,
        "row": row,
        "fecha": fecha_formt,
        "banda": banda
    }
#Parte B: funcion clasificar_banda
def clasificar_banda(banda):
  tablabanda= {
      "B1": "Aerosol costero",
      "B2": "Azul",
      "B3": "Verde",
      "B4": "Rojo",
      "B5": "Infrarrojo cercano (NIR)",
      "B6": "SWIR1",
      "B7": "SWIR2"
  }
  return tablabanda.get(banda, "Desconocida")

# Parte C: procesar todos los archivos
fecha_unicas = set()
print("Procesando archivos:")
for archivo in archivos:
    datos = procesar_nombre_landsat(archivo)
    fecha = datos["fecha"]
    path = datos["path"]
    row = datos["row"]
    banda = clasificar_banda(datos["banda"])
    fecha_unicas.add(fecha)
    print(f"Fecha: {fecha}, Path: {path}, Row: {row}, Banda: {banda}")

print(f"\nCantidad de fechas únicas: {len(fecha_unicas)}")

Procesando archivos:
Fecha: 2024-06-15, Path: 008, Row: 067, Banda: Rojo
Fecha: 2024-06-15, Path: 008, Row: 067, Banda: Infrarrojo cercano (NIR)
Fecha: 2024-07-01, Path: 008, Row: 067, Banda: Verde
Fecha: 2024-06-15, Path: 009, Row: 067, Banda: Rojo
Fecha: 2024-07-08, Path: 008, Row: 067, Banda: SWIR1
Fecha: 2024-07-01, Path: 008, Row: 067, Banda: Rojo

Cantidad de fechas únicas: 3


---
## Problema 2: Inventario Forestal en Madre de Dios (10 puntos)

El **SERFOR** realiza inventarios forestales en Madre de Dios. Los datos incluyen valores faltantes (`-999`) y mediciones con posibles errores.

**Parte A (3 pts):** Funcion `calcular_area_basal(dap_cm)`:
- Devuelve AB en m2: $AB = \pi 	\times  (DAP/200)^2$
- Devuelve `None` si DAP <= 0 o == -999

**Parte B (3 pts):** Funcion `clasificar_arbol(dap_cm, altura_m)` que devuelva:
- `clase`: "Brinzal" (<10cm), "Latizal" (10-25cm), "Fustal menor" (25-50cm), "Fustal mayor" (>=50cm)
- `alerta`: True si DAP > 200cm, altura > 60m, o altura < 1m con DAP > 10cm

**Parte C (4 pts):** Procesa los datos:
1. Para cada árbol, calcule el área basal y clasifíquelo.
2. Omita los árboles con datos faltantes (valores -999).
3. Imprima una advertencia para los árboles marcados.
4. Calcule e imprima las estadísticas descriptivas:
- Número total de árboles válidos
- Área basal total (suma de todos los árboles válidos)
- Cantidad de árboles en cada clase de tamaño
- Número de registros marcados

In [11]:
import math

# Parte A
def calcular_area_basal(d):
    if d==None or d<1 or d==-999:
        return None
    ab = math.pi * ((d/200)**2)
    return ab

# Parte B
def clasificar_arbol(d, h):
    c = ""
    if d < 10:
        c = "Brinzal"
    if d >= 10 and d < 25:
        c = "Latizal"
    if d >= 25 and d < 50:
        c = "Fustal menor"
    if d >= 50:
        c = "Fustal mayor"

    a = False
    if d > 200 or h > 60 or (h < 1 and d > 10):
        a = True

    return c, a

# Parte C
datos = [
    {"id": 1, "dap": 8.5, "alt": 2.2},
    {"id": 2, "dap": 18.0, "alt": 12.5},
    {"id": 3, "dap": 35.0, "alt": 22.0},
    {"id": 4, "dap": 62.0, "alt": 31.0},
    {"id": 5, "dap": -999, "alt": 15.0},
    {"id": 6, "dap": 215.0, "alt": 45.0},
    {"id": 7, "dap": 42.0, "alt": 0.8},
    {"id": 8, "dap": 58.0, "alt": 65.0}
]

tot_v = 0
ab_sum = 0.0
dic_c = {"Brinzal": 0, "Latizal": 0, "Fustal menor": 0, "Fustal mayor": 0}
marcados = 0

print("ADVERTENCIAS:")
for x in datos:
    dp = x["dap"]
    al = x["alt"]
    res_ab = calcular_area_basal(dp)

    if res_ab == None:
        continue

    clase_arbol, es_alerta = clasificar_arbol(dp, al)

    tot_v = tot_v + 1
    ab_sum = ab_sum + res_ab
    dic_c[clase_arbol] = dic_c[clase_arbol] + 1

    if es_alerta == True:
        marcados = marcados + 1
        print("Advertencia en arbol con ID " + str(x["id"]) + ": DAP=" + str(dp) + " Altura=" + str(al))

print("\nESTADISTICAS:")
print("Total de arboles validos:", tot_v)
print("Area basal total:", ab_sum)
print("Arboles por clase:")
for k in dic_c:
    print(k + ": " + str(dic_c[k]))
print("Registros marcados:", marcados)

ADVERTENCIAS:
Advertencia en arbol con ID 6: DAP=215.0 Altura=45.0
Advertencia en arbol con ID 7: DAP=42.0 Altura=0.8
Advertencia en arbol con ID 8: DAP=58.0 Altura=65.0

ESTADISTICAS:
Total de arboles validos: 7
Area basal total: 4.462494919745706
Arboles por clase:
Brinzal: 1
Latizal: 1
Fustal menor: 2
Fustal mayor: 3
Registros marcados: 3


---
## Lista de verificacion
- [ ] Todas las celdas corren sin errores
- [ ] Ambos problemas estan completos
- [ ] Salidas visibles en todas las celdas
- [ ] Nombre incluido